In [1]:
!pip install gymnasium pygame pandas numpy

In [3]:
import random
import gymnasium as gym

env = gym.make('Taxi-v4')

q = {}
for s in range(env.observation_space.n):
    for a in range(env.action_space.n):
        q[(s, a)] = 0.0

def update_q_table(prev_state, action, reward, nextstate, alpha, gamma):
    qa = max([q[(nextstate, a)] for a in range(env.action_space.n)])
    q[(prev_state, action)] += alpha * (reward + gamma * qa - q[(prev_state, action)])

def epsilon_greedy_policy(state, epsilon):
    if random.uniform(0, 1) < epsilon:
        return env.action_space.sample()
    else:
        return max(list(range(env.action_space.n)), key=lambda x: q[(state, x)])

alpha = 0.4
gamma = 0.999
epsilon = 0.017
num_episodes = 2000

for i in range(num_episodes):
    r = 0
    reset_val = env.reset()
    prev_state = reset_val[0] if isinstance(reset_val, tuple) else reset_val

    while True:
        action = epsilon_greedy_policy(prev_state, epsilon)

        step_results = env.step(action)
        if len(step_results) == 5:
            nextstate, reward, terminated, truncated, _ = step_results
            done = terminated or truncated
        else:
            nextstate, reward, done, _ = step_results

        update_q_table(prev_state, action, reward, nextstate, alpha, gamma)
        prev_state = nextstate
        r += reward

        if done:
            break

    if (i + 1) % 200 == 0:
        print(f"Episode {i+1}/{num_episodes} - Total Reward: {r}")

env.close()
print("Training Finished!")


Episode 200/2000 - Total Reward: -10
Episode 400/2000 - Total Reward: 9
Episode 600/2000 - Total Reward: 7
Episode 800/2000 - Total Reward: 11
Episode 1000/2000 - Total Reward: 11
Episode 1200/2000 - Total Reward: 4
Episode 1400/2000 - Total Reward: 5
Episode 1600/2000 - Total Reward: 8
Episode 1800/2000 - Total Reward: 4
Episode 2000/2000 - Total Reward: 2
Training Finished!


In [4]:
import gymnasium as gym
import random
import pandas as pd
import numpy as np

env = gym.make('FrozenLake-v1', is_slippery=True)
alpha = 0.85
gamma = 0.90
epsilon = 0.8
num_episodes = 10000
num_timesteps = 100

def random_policy():
    return env.action_space.sample()

def epsilon_greedy(state, epsilon_val, q_table):
    if random.uniform(0, 1) < epsilon_val:
        return env.action_space.sample()
    else:
        return max(list(range(env.action_space.n)), key=lambda x: q_table.get((state, x), 0.0))

def get_clean_state(reset_output):
    return reset_output[0] if isinstance(reset_output, tuple) else reset_output

def unpack_step(step_output):
    if len(step_output) == 5:
        return step_output[0], step_output[1], (step_output[2] or step_output[3])
    return step_output[0], step_output[1], step_output[2]

print("Running TD Prediction...")
v = {s: 0.0 for s in range(env.observation_space.n)}

for i in range(num_episodes):
    s = get_clean_state(env.reset())
    for t in range(num_timesteps):
        a = random_policy()
        s_next, r, done = unpack_step(env.step(a))
        v[s] += alpha * (r + gamma * v[s_next] - v[s])
        s = s_next
        if done:
            break

df_td_prediction = pd.DataFrame(list(v.items()), columns=['state', 'value'])

print("Running SARSA Control...")
Q_sarsa = {(s, a): 0.0 for s in range(env.observation_space.n) for a in range(env.action_space.n)}

for i in range(num_episodes):
    s = get_clean_state(env.reset())
    a = epsilon_greedy(s, epsilon, Q_sarsa)
    for t in range(num_timesteps):
        s_next, r, done = unpack_step(env.step(a))
        a_next = epsilon_greedy(s_next, epsilon, Q_sarsa)
        Q_sarsa[(s, a)] += alpha * (r + gamma * Q_sarsa[(s_next, a_next)] - Q_sarsa[(s, a)])
        s = s_next
        a = a_next
        if done:
            break

df_sarsa = pd.DataFrame(list(Q_sarsa.items()), columns=['state_action', 'value'])

print("Running Q-Learning...")
Q_learning = {(s, a): 0.0 for s in range(env.observation_space.n) for a in range(env.action_space.n)}

for i in range(num_episodes):
    s = get_clean_state(env.reset())
    for t in range(num_timesteps):
        a = epsilon_greedy(s, epsilon, Q_learning)
        s_next, r, done = unpack_step(env.step(a))
        a_best = np.argmax([Q_learning[(s_next, act)] for act in range(env.action_space.n)])
        Q_learning[(s, a)] += alpha * (r + gamma * Q_learning[(s_next, a_best)] - Q_learning[(s, a)])
        s = s_next
        if done:
            break

df_q_learning = pd.DataFrame(list(Q_learning.items()), columns=['state_action', 'value'])
env.close()

print("\n--- TD Prediction Outputs (First 5 Rows) ---")
print(df_td_prediction.head())
print("\n--- Q-Learning Performance Vectors (First 5 Rows) ---")
print(df_q_learning.head())


Running TD Prediction...
Running SARSA Control...
Running Q-Learning...

--- TD Prediction Outputs (First 5 Rows) ---
   state     value
0      0  0.000096
1      1  0.000013
2      2  0.000341
3      3  0.000016
4      4  0.000183

--- Q-Learning Performance Vectors (First 5 Rows) ---
  state_action     value
0       (0, 0)  0.409408
1       (0, 1)  0.299048
2       (0, 2)  0.298225
3       (0, 3)  0.395964
4       (1, 0)  0.000145
